In [ ]:
!pip install langchain langchain-groq python-dotenv

**State**

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
  messages: Annotated[list, add_messages]

# Translation: "State has a 'messages' list.
# When new info comes, ADD it, don't overwrite."

**Workers or Nodes**

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

In [ ]:
from langchain_groq import ChatGroq

# Setup the "Brain" — LLaMA 3.3 running on Groq's servers
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,                                      # 0 = factual, 1 = creative
    api_key=""                                  # Get free key from console.groq.com
)

# Hand the calculator to the student
llm_with_tools = llm.bind_tools([multiply])

In [ ]:
from langgraph.prebuilt import ToolNode

# Worker or Node 1: The Assistant (asks LLM to think)
def assistant_node(state: State):
  response = llm_with_tools.invoke(state["messages"])
  return {"messages": [response]}

# Worker or Node 2: The Tool Runner (executes tool calls)
tool_node = ToolNode([multiply])

**Edge or Route**

In [ ]:
from langgraph.graph import END

def should_continue(state: State):
  last_msg = state["messages"][-1]
  if last_msg.tool_calls:                              # LLM wants a tool?
      return "tools"                                   # → Go to tool node
  return END

**Creating Final Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END

#state
builder = StateGraph(State)

#nodes
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

#edge
builder.add_edge(START, "assistant")       # Start → Assistant
builder.add_conditional_edges(            # Router logic
    "assistant", should_continue
)
builder.add_edge("tools", "assistant")     # Tools → back

graph = builder.compile()

**Using Agent**

In [ ]:
query = "Calculate 50 times 173 and then divide by 5."
result = graph.invoke({"messages": [("user", query)]})
print(result["messages"][-1].content)

The result of 50 times 173 is 8650. Then, dividing 8650 by 5 results in 1730.


**Crypto Checker**

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,      # 0 = factual, 1 = creative
    api_key="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"  # Free from console.groq.com
)

In [ ]:
import requests
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def get_crypto_price(coin_id: str) -> float:
    """Fetches current price of a cryptocurrency in USD.
    Args: coin_id: e.g., 'bitcoin', 'ethereum'"""
    url = f"https://api.coingecko.com/api/v3/simple/price?ids={coin_id}&vs_currencies=usd"
    data = requests.get(url).json()
    return data[coin_id]['usd']

In [ ]:
tools = [get_crypto_price, multiply]
llm_with_tools = llm.bind_tools(tools)

initial_state = "I own 3 bitcoin. Check the live price and tell me my portfolio value in USD."

response = llm_with_tools.invoke(initial_state)
print(f"Tool Calls: {response.tool_calls}")

Tool Calls: [{'name': 'get_crypto_price', 'args': {'coin_id': 'bitcoin'}, 'id': '0bv4pwh0q', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 3, 'b': 12345}, 'id': '4pett53ng', 'type': 'tool_call'}]


**Using Langgraph**

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
  messages: Annotated[list, add_messages]

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import END

tools = [get_crypto_price, multiply]

# Worker 1: The Assistant (asks LLM to think)
def assistant_node(state: State):
  response = llm_with_tools.invoke(state["messages"])
  return {"messages": [response]}

# Worker 2: The Tool Runner (executes tool calls)
tool_node = ToolNode(tools)

# The Router: Traffic light logic
def should_continue(state: State):
  last_msg = state["messages"][-1]
  if last_msg.tool_calls:                                     # LLM wants a tool?
      return "tools"                                          # → Go to tool node
  return END                                                  # → Done, stop the loop

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")       # Start → Assistant
builder.add_conditional_edges(            # Router logic
    "assistant", should_continue
)
builder.add_edge("tools", "assistant")     # Tools → back

graph = builder.compile()

In [ ]:
initial_state = {
 "messages": [
 ("user", "I own 3 bitcoin. Check the live price and tell me my portfolio value in USD along with live price.")
 ]
}
result = graph.invoke(initial_state)
print(result["messages"][-1].content)

The live price of Bitcoin is $43,900. Your portfolio value is $68,091.
